
<div dir="rtl" style="text-align:right">

# Phase 5 — Vector Search
### جزوه‌ی آموزشی مرحله‌به‌مرحله با مثال‌های عملی

در این نوت‌بوک با مفاهیم اصلی جست‌وجوی برداری آشنا می‌شویم و یک نمونه‌ی کوچک از سیستم بازیابی می‌سازیم. مثال‌ها از چند جمله‌ی فارسی استفاده می‌کنند؛ برای سادگی، بردارها را به‌صورت دستی تعریف می‌کنیم تا تمرکز روی خودِ Vector Search باشد، نه ساخت Embedding.

**پیش‌نیاز:** آشنایی مقدماتی با Python و آرایه‌ها.  
**کتابخانه‌های اصلی:** `numpy` و `scikit-learn`. بخش‌های FAISS و Chroma اختیاری‌اند و نصب جداگانه دارند.

</div>


In [10]:
from IPython.display import HTML, display

display(
    HTML("""
<style>
.jp-RenderedHTMLCommon, .jp-MarkdownOutput {
    direction: rtl;
    text-align: right;
    font-family: Tahoma, 'Segoe UI', sans-serif;
    line-height: 1.9;
}
.jp-RenderedHTMLCommon table { direction: rtl; }
.jp-CodeCell .cm-editor, .jp-CodeCell pre, .jp-InputArea-editor {
    direction: ltr !important;
    text-align: left !important;
}
</style>
""")
)


<div dir="rtl" style="text-align:right">

## ۱. Vector Search Fundamentals — مبانی جست‌وجوی برداری

در جست‌وجوی سنتی معمولاً کلمات یا عبارت‌ها را با هم تطبیق می‌دهیم. اما در **Vector Search** هر متن، تصویر یا داده‌ی دیگر به یک بردار عددی (Embedding) تبدیل می‌شود. سپس به‌جای تطبیق دقیق کلمات، نزدیکی بردارها را اندازه می‌گیریم.

- **Embedding:** نمایش عددی یک داده در فضای چندبعدی.
- **Vector:** آرایه‌ای از اعداد، مانند `[0.2, 0.7, 0.1]`.
- **Distance / Similarity:** معیاری برای سنجش فاصله یا شباهت دو بردار.

در مثال زیر، بردارها فرضی‌اند و صرفاً برای آموزش ساخته شده‌اند؛ در پروژه‌ی واقعی باید آن‌ها را با یک مدل Embedding تولید کنید.

</div>


In [ ]:
import numpy as np

documents = [
    "آموزش پایتون برای برنامه‌نویسان",
    "یادگیری ماشین و هوش مصنوعی",
    "آموزش آشپزی و غذاهای ایرانی",
    "مبانی شبکه و زیرساخت نرم‌افزار",
]

# Embeddingهای فرضی و دوبعدی برای نمایش مفهوم
vectors = np.array(
    [
        [0.95, 0.10],  # برنامه‌نویسی
        [0.82, 0.55],  # هوش مصنوعی
        [0.05, 0.95],  # آشپزی
        [0.55, 0.25],  # شبکه
    ],
    dtype=np.float32,
)

query_vector = np.array([0.90, 0.30], dtype=np.float32)

print("تعداد اسناد:", len(documents))
print("ابعاد هر بردار:", vectors.shape[1])


تعداد اسناد: 4
ابعاد هر بردار: 2



<div dir="rtl" style="text-align:right">

## ۲. Exact Nearest Neighbor Search — جست‌وجوی دقیق نزدیک‌ترین همسایه

در روش Exact، فاصله‌ی Query با **تمام** بردارهای موجود محاسبه می‌شود؛ بنابراین نتیجه دقیق است، اما با بزرگ شدن مجموعه‌داده هزینه‌ی محاسباتی افزایش می‌یابد.

سه معیار رایج:

- **Euclidean Distance (L2):** فاصله‌ی هندسی دو بردار؛ مقدار کمتر یعنی نزدیک‌تر.
- **Cosine Similarity:** زاویه‌ی بین دو بردار؛ مقدار بیشتر یعنی مشابه‌تر.
- **Dot Product:** حاصل‌ضرب داخلی؛ در بعضی مدل‌ها و با فرض‌های مشخص کاربرد دارد.

در این مثال از Cosine Similarity استفاده می‌کنیم. این معیار به بزرگی بردارها حساسیت کمتری دارد و جهت آن‌ها را مقایسه می‌کند.

</div>


In [ ]:
def cosine_similarity(query, matrix):
    query_norm = np.linalg.norm(query)
    matrix_norms = np.linalg.norm(matrix, axis=1)
    # جلوگیری از تقسیم بر صفر
    denominator = np.maximum(query_norm * matrix_norms, 1e-12)
    return (matrix @ query) / denominator


scores = cosine_similarity(query_vector, vectors)
ranked_indices = np.argsort(scores)[::-1]

for idx in ranked_indices:
    print(f"{scores[idx]:.3f} | {documents[idx]}")


0.995 | مبانی شبکه و زیرساخت نرم‌افزار
0.977 | آموزش پایتون برای برنامه‌نویسان
0.964 | یادگیری ماشین و هوش مصنوعی
0.366 | آموزش آشپزی و غذاهای ایرانی



<div dir="rtl" style="text-align:right">

## ۳. Approximate Nearest Neighbor (ANN) Search — جست‌وجوی تقریبی

وقتی تعداد بردارها بسیار زیاد شود، بررسی تک‌تک آن‌ها ممکن است کند باشد. الگوریتم‌های **ANN** برای کاهش زمان جست‌وجو، معمولاً بخشی از فضای جست‌وجو را بررسی می‌کنند. در نتیجه ممکن است سریع‌تر باشند، اما تضمین نمی‌کنند دقیقاً همان همسایه‌های روش Exact را برگردانند.

معیار مهم ارزیابی ANN، **Recall@K** است: چه سهمی از K نتیجه‌ی دقیق، در K نتیجه‌ی تقریبی نیز پیدا شده‌اند؟

نمونه‌ی عملی با FAISS و ایندکس HNSW در سلول بعدی اختیاری است. برای نصب در محیط Jupyter از دستور زیر استفاده کنید. اگر نصب انجام نشد، می‌توانید ادامه‌ی نوت‌بوک را با روش Exact اجرا کنید.

</div>


In [4]:
# اجرای اختیاری ANN با FAISS (پس از نصب موفق)
try:
    import faiss

    # FAISS برای جست‌وجوی L2 در این مثال از بردارهای float32 استفاده می‌کند.
    xb = vectors.astype("float32")
    index_hnsw = faiss.IndexHNSWFlat(xb.shape[1], 8)  # گراف HNSW
    index_hnsw.hnsw.efConstruction = 40
    index_hnsw.hnsw.efSearch = 32
    index_hnsw.add(xb)

    distances, ids = index_hnsw.search(query_vector.reshape(1, -1), 3)
    print("نتایج ANN:")
    for distance, idx in zip(distances[0], ids[0]):
        if idx >= 0:
            print(f"L2={distance:.3f} | {documents[idx]}")
except ImportError:
    print("FAISS نصب نیست؛ سلول نصب را اجرا کنید یا از Exact Search استفاده کنید.")


نتایج ANN:
L2=0.043 | آموزش پایتون برای برنامه‌نویسان
L2=0.069 | یادگیری ماشین و هوش مصنوعی
L2=0.125 | مبانی شبکه و زیرساخت نرم‌افزار



<div dir="rtl" style="text-align:right">

## ۴. Vector Indexes — ایندکس‌های برداری

ایندکس ساختاری کمکی است که جست‌وجو در مجموعه‌داده را سریع‌تر می‌کند. انتخاب ایندکس به اندازه‌ی داده، حافظه، سرعت موردنیاز و میزان دقت قابل‌قبول بستگی دارد.

| نوع ایندکس | ایده‌ی کلی | نکته |
|---|---|---|
| Flat / Brute Force | بررسی همه‌ی بردارها | دقیق، ساده، معمولاً کندتر در مقیاس بزرگ |
| HNSW | گراف چندلایه برای پیمایش همسایه‌ها | جست‌وجوی سریع تقریبی؛ مصرف حافظه مهم است |
| IVF | تقسیم فضا به خوشه‌ها و جست‌وجو در چند خوشه | سرعت با تعداد خوشه‌های بررسی‌شده trade-off دارد |
| PQ | فشرده‌سازی بردارها | کاهش حافظه با احتمال افت دقت |

**نکته:** «ایندکس» با «دیتابیس برداری» یکی نیست. ایندکس ساختار جست‌وجو است؛ دیتابیس علاوه بر جست‌وجو، قابلیت‌هایی مانند ذخیره‌سازی، مدیریت داده و فیلتر متادیتا را هم ارائه می‌کند.

</div>



<div dir="rtl" style="text-align:right">

## ۵. Metadata Filtering — فیلتر کردن با متادیتا

متادیتا اطلاعات توصیفی هر سند است؛ مانند دسته‌بندی، زبان، تاریخ یا شناسه‌ی منبع. گاهی ابتدا اسناد را با شرط متادیتا محدود می‌کنیم و سپس جست‌وجوی برداری را روی همان اسناد انجام می‌دهیم.

در این مثال فقط اسناد دسته‌ی `technology` را وارد جست‌وجو می‌کنیم.

</div>


In [5]:
metadata = [
    {"category": "technology", "language": "fa"},
    {"category": "technology", "language": "fa"},
    {"category": "cooking", "language": "fa"},
    {"category": "technology", "language": "fa"},
]

allowed = np.array([m["category"] == "technology" for m in metadata])
filtered_vectors = vectors[allowed]
filtered_docs = [doc for doc, keep in zip(documents, allowed) if keep]

filtered_scores = cosine_similarity(query_vector, filtered_vectors)
filtered_order = np.argsort(filtered_scores)[::-1]

for idx in filtered_order:
    print(f"{filtered_scores[idx]:.3f} | {filtered_docs[idx]}")


0.995 | مبانی شبکه و زیرساخت نرم‌افزار
0.977 | آموزش پایتون برای برنامه‌نویسان
0.964 | یادگیری ماشین و هوش مصنوعی



<div dir="rtl" style="text-align:right">

## ۶. Top-K Retrieval — بازیابی K نتیجه‌ی برتر

در بسیاری از سیستم‌های RAG لازم نیست تمام اسناد مرتبط برگردانده شوند. معمولاً فقط K نتیجه‌ی برتر را انتخاب می‌کنیم.

- `K=3` یعنی سه نتیجه‌ی اول.
- K خیلی کوچک ممکن است اطلاعات مرتبط را حذف کند.
- K خیلی بزرگ می‌تواند نویز و هزینه‌ی پردازش مرحله‌ی بعد را افزایش دهد.

</div>


In [ ]:
def top_k_search(query, matrix, texts, k=3):
    scores = cosine_similarity(query, matrix)
    # محدود کردن K به تعداد اسناد موجود
    k = min(max(k, 0), len(texts))
    indices = np.argsort(scores)[::-1][:k]
    return [(texts[i], float(scores[i])) for i in indices]


for text, score in top_k_search(query_vector, vectors, documents, k=2):
    print(f"{score:.3f} | {text}")


0.995 | مبانی شبکه و زیرساخت نرم‌افزار
0.977 | آموزش پایتون برای برنامه‌نویسان



<div dir="rtl" style="text-align:right">

## ۷. Similarity Thresholds — آستانه‌ی شباهت

Top-K همیشه دقیقاً K نتیجه برمی‌گرداند؛ حتی اگر همه‌ی نتایج نامرتبط باشند. با **Similarity Threshold** می‌توانیم نتایجی را که امتیازشان از یک حد کمتر است حذف کنیم.

آستانه‌ی مناسب مقدار ثابتی برای همه‌ی پروژه‌ها نیست. باید آن را با داده‌ی واقعی و ارزیابی کیفیت بازیابی تنظیم کنید. همچنین آستانه به مدل Embedding و معیار شباهت وابسته است.

</div>


In [ ]:
def threshold_search(query, matrix, texts, threshold=0.75):
    scores = cosine_similarity(query, matrix)
    indices = np.argsort(scores)[::-1]
    return [(texts[i], float(scores[i])) for i in indices if scores[i] >= threshold]


results = threshold_search(query_vector, vectors, documents, threshold=0.75)
if results:
    for text, score in results:
        print(f"{score:.3f} | {text}")
else:
    print("نتیجه‌ای بالاتر از آستانه پیدا نشد.")


0.995 | مبانی شبکه و زیرساخت نرم‌افزار
0.977 | آموزش پایتون برای برنامه‌نویسان
0.964 | یادگیری ماشین و هوش مصنوعی



<div dir="rtl" style="text-align:right">

## ۸. Vector Database Fundamentals — مبانی دیتابیس برداری

دیتابیس برداری برای ذخیره و بازیابی داده‌های برداری طراحی شده است. معمولاً هر رکورد شامل این بخش‌هاست:

1. **ID:** شناسه‌ی یکتا
2. **Vector / Embedding:** بردار عددی
3. **Metadata:** اطلاعات توصیفی و قابل فیلتر
4. **Document / Payload:** متن یا محتوای اصلی (در صورت پشتیبانی سیستم)

در یک معماری RAG، روند ساده به این شکل است:

1. اسناد را دریافت و به قطعه‌های کوچک‌تر (Chunk) تقسیم می‌کنیم.
2. برای هر Chunk یک Embedding می‌سازیم.
3. بردار، شناسه، متن و متادیتا را ذخیره می‌کنیم.
4. سؤال کاربر را به Embedding تبدیل می‌کنیم.
5. با Vector Search اسناد مرتبط را بازیابی می‌کنیم.
6. اسناد بازیابی‌شده را به‌عنوان Context به LLM می‌دهیم.

### مثال اختیاری با Chroma

این بخش یک نمونه‌ی حداقلی از ذخیره‌سازی و Query در Chroma است. Chroma در این مثال خودش Embedding تولید می‌کند؛ برای پروژه‌ی واقعی می‌توانید تابع Embedding دلخواه خودتان را به آن متصل کنید.

</div>


In [20]:
try:
    import chromadb

    client = chromadb.Client()
    collection_name = "phase5_demo"

    try:
        client.delete_collection(name=collection_name)
    except Exception:
        pass

    collection = client.create_collection(name=collection_name)

    collection.add(
        ids=["doc-1", "doc-2", "doc-3"],
        documents=[
            "آموزش پایتون و برنامه‌نویسی",
            "یادگیری ماشین و مفاهیم هوش مصنوعی",
            "آموزش آشپزی و غذاهای ایرانی",
        ],
        metadatas=[
            {"category": "technology"},
            {"category": "technology"},
            {"category": "cooking"},
        ],
    )
    result = collection.query(
        query_texts=["مفاهیم هوش مصنوعی"],
        n_results=2,
        where={"category": "technology"},
    )
    print(result["ids"])
    print(result["documents"])
except ImportError:
    print("Chroma نصب نیست؛ سلول نصب را اجرا کنید.")

[['doc-2', 'doc-1']]
[['یادگیری ماشین و مفاهیم هوش مصنوعی', 'آموزش پایتون و برنامه\u200cنویسی']]



<div dir="rtl" style="text-align:right">

## جمع‌بندی و تمرین

در این Phase باید بتوانید تفاوت Exact و ANN را توضیح دهید، معیار شباهت را انتخاب کنید، Top-K و Threshold را به‌کار ببرید و نقش Index، Metadata و Vector Database را بشناسید.

### تمرین‌های کوتاه

1. مقدار `k` را از ۲ به ۳ تغییر دهید؛ چه تفاوتی در خروجی می‌بینید؟
2. آستانه را از `0.75` به `0.90` تغییر دهید؛ چند نتیجه باقی می‌ماند؟
3. یک سند جدید به `documents` و `vectors` اضافه کنید و دوباره جست‌وجو بگیرید.
4. فیلتر را طوری تغییر دهید که فقط دسته‌ی `cooking` انتخاب شود.
5. برای یک دیتاست واقعی، نتایج Exact و ANN را با معیار Recall@K مقایسه کنید.

### منابع رسمی برای مطالعه‌ی بیشتر

- [FAISS — مستندات و راهنمای شروع](https://github.com/facebookresearch/faiss/wiki)
- [Chroma — مستندات رسمی](https://docs.trychroma.com/)
- [scikit-learn — Nearest Neighbors](https://scikit-learn.org/stable/modules/neighbors.html)
- [NumPy — مستندات رسمی](https://numpy.org/doc/stable/)

</div>
